In [72]:
import sys
import numpy as np
import pandas as pd
 
if 'google.colab' in sys.modules:
    %pip install pyomo >/dev/null 2>/dev/null
    %pip install highspy >/dev/null 2>/dev/null
 
solver = 'appsi_highs'
 
import pyomo.environ as pyo
SOLVER = pyo.SolverFactory(solver)
 
assert SOLVER.available(), f"Solver {solver} is not available."

## ✅ The microchip production problem

### Problem description

The company BIM (Best International Machines) produces two types of microchips, logic chips (1g silicon, 1g plastic, 4g copper) and memory chips (1g germanium, 1g plastic, 2g copper). Each of the logic chips can be sold for a 12€ profit, and each of the memory chips for a 9€ profit. The current stock of raw materials is as follows: 1000g silicon, 1500g germanium, 1750g plastic, 4800g copper. How many microchips of each type should be produced to maximize profit while respecting the availability of raw material stock?


| Product (P)   | Revenue |
| ------------- | ------- |
| log_chips (l) | 12      |
| mem_chips (m) | 9       |

---

| Product   | Silicone | Plastic | Copper | Germanium |
| --------- | -------- | ------- | ------ | --------- |
| log_chips | 1        | 1       | 4      | 0         |
| mem_chips | 0        | 1       | 2      | 1         |

---

| Resource  | Available |
| --------- | --------- |
| Silicone  | 1000      |
| Germanium | 1500      |
| Plastic   | 1750      |
| Copper    | 4800      |

---


### Create the mathematical model

Obj:
\begin{equation}
\max_{P_p} \; Profit = \sum_{p \in \{l,m\}} \left( Revenue_p - Cost_p \right) \cdot P_p
\end{equation}
Constraints:
\begin{equation}
\sum_{p \in \{l,m\}} Resource_{r,p} \cdot P_p \;\le\; a_r
\qquad \forall r \in \{Si,Pl,Cu,Gr\}.
\end{equation}


In [73]:
# First as usual create Model 
model=pyo.ConcreteModel()
# Then add sets 
products = {
    "l":{"Si":1,
         "Pl":1,
         "Cu":4,
         "Gr":0},
    "m":{"Si":0,
         "Pl":1,
         "Cu":2,
         "Gr":1}
}

product_price = {
     "l":12,
     "m":9
}

resources={
    "Si":1000,
    "Pl":1750,
    "Cu":4800,
    "Gr":1500
     }

model.PRODUCTS = pyo.Set(initialize = products.keys())
model.RESOURCES = pyo.Set(initialize = resources.keys())

# Add Parameters / Exogeneus values!
#Availibility
@model.Param(model.RESOURCES, domain=pyo.Any)
def availibility(model,material):
     return resources[material]

# Process a[p,r] 
@model.Param(model.PRODUCTS,model.RESOURCES, domain=pyo.Any)
def process(model,product,material):
     return products[product][material]
# Price of products
@model.Param(model.PRODUCTS,domain=pyo.Any)
def prices(model,product):
     return product_price[product]


In [74]:
# Pyomo calls your bounds function (lambda function) once per index element while it creates each scalar variable R[resource]

# Create Decision variables. In this problem, variable P_l and P_l for quantity of product
model.P = pyo.Var(
    model.PRODUCTS,
    bounds = (0,None),
    domain=pyo.Integers
)
model.R = pyo.Var(
    model.RESOURCES,# Set used as index for variables
    bounds = lambda m, r: (0, model.availibility[r]),# Rule that returns tuples with upper and lower bound
    domain=pyo.Integers
)


In [75]:
#create objective function
#function to optimize
model.revenue = pyo.quicksum(
     model.P[p] * model.prices [p] for p in model.PRODUCTS
)
@model.Objective(sense=pyo.maximize)
def profit(model):
     return model.revenue


In [78]:
# Create constraints
@model.Constraint(model.RESOURCES)
def materials_used(model, resource):
    # calculate how much resources are used up for each product
    return (
        pyo.quicksum(
            model.process[product, resource] * model.P[product] for product in model.PRODUCTS
        )
        <= model.R[resource]
    )

In [79]:
model.pprint()

2 Set Declarations
    PRODUCTS : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'l', 'm'}
    RESOURCES : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'Si', 'Pl', 'Cu', 'Gr'}

3 Param Declarations
    availibility : Size=4, Index=RESOURCES, Domain=Any, Default=None, Mutable=False
        Key : Value
         Cu :  4800
         Gr :  1500
         Pl :  1750
         Si :  1000
    prices : Size=2, Index=PRODUCTS, Domain=Any, Default=None, Mutable=False
        Key : Value
          l :    12
          m :     9
    process : Size=8, Index=PRODUCTS*RESOURCES, Domain=Any, Default=None, Mutable=False
        Key         : Value
        ('l', 'Cu') :     4
        ('l', 'Gr') :     0
        ('l', 'Pl') :     1
        ('l', 'Si') :     1
        ('m', 'Cu') :     2
        ('m', 'Gr') :     1
        ('m', 'Pl') :     1
        ('m

In [82]:
# solve
SOLVER.solve(model)

{'Problem': [{'Lower bound': 17700.0, 'Upper bound': 17700.0, 'Number of objectives': 1, 'Number of constraints': 0, 'Number of variables': 0, 'Sense': 'maximize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Termination message': 'TerminationCondition.optimal'}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

In [103]:
print("Profit",model.profit())

print("\nProduction Report")
for product in model.PRODUCTS:
    print(f" {product}  produced =  {pyo.value(model.P[product])}")

print("\nResource Report")
for resource in model.RESOURCES:
    print(f" {resource} consumed = {pyo.value(model.R[resource])}")

Profit 17700.0

Production Report
 l  produced =  650.0
 m  produced =  1100.0

Resource Report
 Si consumed = 1000.0
 Pl consumed = 1750.0
 Cu consumed = 4800.0
 Gr consumed = 1500.0


### Faster Solution by building the matrices first:


In [105]:
import numpy as np

model = pyo.ConcreteModel("BIM production planning in matrix form")

# Define the number of variables and constraints
n_vars = 2
n_constraints = 4

# Decision variables and their domain
model.x = pyo.Var(range(n_vars), domain=pyo.NonNegativeReals)

# Define the vectors and matrices
c = np.array([-12, -9])
A = np.array([[-1, 0], [0, -1], [-1, -1], [-4, -2]])
b = np.array([-1000, -1500, -1750, -4800])

# Objective function
model.profit = pyo.Objective(
    expr=sum(c[i] * model.x[i] for i in range(n_vars)), sense=pyo.minimize
)

# Constraints
model.constraints = pyo.ConstraintList()
for i in range(n_constraints):
    model.constraints.add(expr=sum(A[i, j] * model.x[j] for j in range(n_vars)) >= b[i])

# Solve and print solution
SOLVER.solve(model)
optimal_x = [pyo.value(model.x[i]) for i in range(n_vars)]
print(f"x = {tuple(np.round(optimal_x, 1))}")
print(f"optimal value = {-pyo.value(model.profit):.1f}")

x = (np.float64(650.0), np.float64(1100.0))
optimal value = 17700.0


## Other Problem